In [ ]:
# # Attention Rollout Analysis 
# Implements three rollout strategies:
#   1. Discard-Ratio Rollout (no gradients — zeros out bottom 90% of attention)
#   2. Gradient-Weighted Rollout (uses backward pass for task-aware attribution)
#   3. Vanilla Rollout (baseline, typically diffuse — kept for comparison)
#
# Based on: jacobgil/vit-explain and hila-chefer/Transformer-Explainability


In [ ]:

from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from scipy.stats import entropy as scipy_entropy
from tqdm import tqdm
from transformers import AutoProcessor

import os
import time
from vla.constants import PROJECT_ROOT
from vla.models.smolvla import SmolVLAPolicy



In [ ]:

suite_name = "object"
target_word = "juice"
frame_range = range(0, 110)  # Frames 0000 to 0109

# Resolve paths
task_dir_name = os.listdir(PROJECT_ROOT / "data/images/libero" / suite_name)[0]
image_dir = PROJECT_ROOT / "data/images/libero" / suite_name / "ep0000_task_0"
task_path = (
    PROJECT_ROOT / "data/images/libero" / suite_name / task_dir_name / "task.txt"
)

with open(task_path) as f:
    task_description = f.read().strip()

print(f"Task: {task_description}")
print(
    f"Frames: {frame_range.start:04d} – {frame_range.stop - 1:04d} "
    f"({len(frame_range)} total)"
)



In [ ]:

checkpoint = "HuggingFaceVLA/smolvla_libero"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading model from checkpoint: {checkpoint} on device: {device}")
policy = SmolVLAPolicy(checkpoint=checkpoint, device=device, action_dim=7)



In [ ]:

hf_vlm = policy.model.vlm_with_expert.vlm
llm = hf_vlm.model.text_model
total_layers = len(llm.layers)
layer_indices = list(range(total_layers // 2))  # SmolVLA discards final 50%
vision_dtype = next(hf_vlm.model.vision_model.parameters()).dtype

# Force eager attention output
llm.config._attn_implementation = "eager"
llm.config.output_attentions = True

# Load processor ONCE
try:
    processor = AutoProcessor.from_pretrained("HuggingFaceVLA/smolvla_libero")
except Exception:
    processor = AutoProcessor.from_pretrained("HuggingFaceTB/SmolVLM2-500M-Instruct")

# Pre-compute the text prompt template (same for every frame)
messages = [
    {
        "role": "user",
        "content": [{"type": "image"}, {"type": "text", "text": task_description}],
    }
]
prompt_template = processor.apply_chat_template(messages, add_generation_prompt=True)

print(f"Model ready — {total_layers} total layers, using first {len(layer_indices)}")
print(f"Processor loaded, prompt template cached.")



In [ ]:
def get_attention_for_image(pil_image, device="cuda"):
    """
    Forward pass only — returns attention maps and inputs.
    """
    attention_maps = {}

    def get_forward_hook(layer_idx):
        def hook(module, input, output):
            attention_maps[layer_idx] = output[1].detach().cpu()
        return hook

    hook_handles = []
    for idx in layer_indices:
        target_layer = llm.layers[idx].self_attn
        handle = target_layer.register_forward_hook(get_forward_hook(idx))
        hook_handles.append(handle)

    inputs = processor(
        text=prompt_template, images=pil_image, return_tensors="pt"
    ).to(device)
    inputs["pixel_values"] = inputs["pixel_values"].to(vision_dtype)

    with torch.no_grad():
        _ = hf_vlm(**inputs)

    for handle in hook_handles:
        handle.remove()

    return attention_maps, inputs



In [ ]:

# %% Forward + backward extraction (for gradient-weighted rollout)
def get_attention_with_gradients(pil_image, target_word, device="cuda"):
    """
    Forward + backward pass to capture attention weights AND their gradients.

    Key insight: to build a computation graph through the attention weights,
    we temporarily enable requires_grad on the embedding layer (embed_tokens).
    This makes all downstream tensors — Q, K, attention_weights, hidden_states —
    part of the autograd graph. We can then call retain_grad() on attention
    weights and read .grad after backward().

    The embedding weight is reset to its original state immediately after the
    backward pass. Only one weight matrix participates in the graph, keeping
    memory overhead minimal.

    Returns:
        attention_maps: dict {layer_idx: tensor(1, H, S, S)}
        attention_grads: dict {layer_idx: tensor(1, H, S, S)}
        inputs: tokenized inputs dict
        target_idx: index of the target token
    """
    attention_maps = {}
    attention_grads = {}
    attn_tensors = {}  # live references (not detached) for .grad access

    def get_forward_hook(layer_idx):
        def hook(module, input, output):
            attn = output[1]          # (1, H, S, S), requires_grad=True now
            attn.retain_grad()        # keep gradient for non-leaf tensor
            attn_tensors[layer_idx] = attn           # live reference in graph
            attention_maps[layer_idx] = attn.detach().cpu()
        return hook

    hook_handles = []
    for idx in layer_indices:
        target_layer = llm.layers[idx].self_attn
        handle = target_layer.register_forward_hook(get_forward_hook(idx))
        hook_handles.append(handle)

    inputs = processor(
        text=prompt_template, images=pil_image, return_tensors="pt"
    ).to(device)
    inputs["pixel_values"] = inputs["pixel_values"].to(vision_dtype)

    # Temporarily enable requires_grad on embed_tokens so that the entire
    # computation chain (embeddings → Q/K → attn_weights → ...) is part of
    # the autograd graph, making attention weight tensors require grad.
    embed_tokens = llm.embed_tokens
    orig_req_grad = embed_tokens.weight.requires_grad
    embed_tokens.weight.requires_grad_(True)

    hf_vlm.zero_grad()
    with torch.enable_grad():
        outputs = hf_vlm(**inputs)

    # Find target token index
    input_ids = inputs["input_ids"][0].tolist()
    tokens = processor.tokenizer.convert_ids_to_tokens(input_ids)
    target_idx = -1
    for i, t in enumerate(tokens):
        if target_word.lower() in t.lower():
            target_idx = i
            break
    if target_idx == -1:
        target_idx = len(input_ids) - 1

    # Scalar loss: norm of the target token's logit vector.
    # Backpropagating this tells the network which attention pathways
    # contributed to building the representation at that token position.
    target_logits = outputs.logits[0, target_idx, :].float()
    loss = target_logits.norm()
    loss.backward()

    # Collect gradients
    for layer_idx, attn_tensor in attn_tensors.items():
        if attn_tensor.grad is not None:
            attention_grads[layer_idx] = attn_tensor.grad.detach().cpu()
        else:
            # Fallback: gradient didn't flow (shouldn't happen with this setup)
            attention_grads[layer_idx] = torch.ones_like(attention_maps[layer_idx])

    # Cleanup — restore embedding to non-grad state, zero grads
    for handle in hook_handles:
        handle.remove()
    embed_tokens.weight.requires_grad_(orig_req_grad)
    hf_vlm.zero_grad()

    return attention_maps, attention_grads, inputs, target_idx



In [ ]:
# %% Strategy 1: Discard-Ratio Rollout (no gradients)
def compute_discard_rollout(attention_maps_dict, layer_indices, discard_ratio=0.9,
                             head_fusion="mean"):
    """
    Attention Rollout with noise filtering via discard ratio.

    At each layer, the bottom `discard_ratio` fraction of attention values
    are zeroed out before the rollout multiplication. This prevents low-
    confidence attention from accumulating noise across layers.

    From: jacobgil/vit-explain

    Args:
        attention_maps_dict: {layer_idx: (1, num_heads, S, S)}
        layer_indices: ordered list of layer indices
        discard_ratio: fraction of lowest attention values to zero out (0.9 = keep top 10%)
        head_fusion: "mean", "max", or "min" across attention heads
    """
    result = None

    for layer_idx in layer_indices:
        attn = attention_maps_dict[layer_idx].squeeze(0)  # (H, S, S)

        if attn.dtype == torch.bfloat16:
            attn = attn.to(torch.float32)

        # Fuse heads
        if head_fusion == "mean":
            fused = attn.mean(dim=0)  # (S, S)
        elif head_fusion == "max":
            fused = attn.max(dim=0)[0]
        elif head_fusion == "min":
            fused = attn.min(dim=0)[0]
        else:
            raise ValueError(f"Unknown head_fusion: {head_fusion}")

        # DISCARD low attention values — this is the key fix
        flat = fused.view(-1)
        k = int(flat.size(0) * discard_ratio)
        if k > 0:
            _, low_indices = flat.topk(k, largest=False)
            flat[low_indices] = 0

        # Reshape back
        seq_len = attn.shape[-1]
        fused = flat.view(seq_len, seq_len)

        # Add identity for residual connection and normalize
        identity = torch.eye(seq_len, dtype=fused.dtype)
        a = (fused + identity) / 2
        a = a / (a.sum(dim=-1, keepdim=True) + 1e-12)

        if result is None:
            result = a
        else:
            result = a @ result

    return result.numpy()


In [ ]:

# %% Strategy 2: Gradient-Weighted Rollout
def compute_grad_rollout(attention_maps_dict, attention_grads_dict, layer_indices,
                          discard_ratio=0.9):
    """
    Gradient-weighted Attention Rollout.

    Instead of simply averaging heads, we weight each head's attention by
    its gradient and apply ReLU (keeping only positive contributions).
    This makes the rollout "task-aware" — only attention pathways that
    actually contributed to the target token survive.

    From: jacobgil/vit-explain (vit_grad_rollout.py)

    Args:
        attention_maps_dict: {layer_idx: (1, H, S, S)}
        attention_grads_dict: {layer_idx: (1, H, S, S)}
        layer_indices: ordered list
        discard_ratio: fraction to zero out after gradient weighting
    """
    result = None

    for layer_idx in layer_indices:
        attn = attention_maps_dict[layer_idx]    # (1, H, S, S)
        grad = attention_grads_dict[layer_idx]   # (1, H, S, S)

        if attn.dtype == torch.bfloat16:
            attn = attn.to(torch.float32)
        if grad.dtype == torch.bfloat16:
            grad = grad.to(torch.float32)

        # Weight attention by gradient then average over heads
        weighted = (attn * grad).squeeze(0)  # (H, S, S)
        fused = weighted.mean(dim=0)         # (S, S)

        # ReLU — only keep positive contributions
        fused = torch.clamp(fused, min=0)

        # Discard low values
        flat = fused.view(-1)
        k = int(flat.size(0) * discard_ratio)
        if k > 0:
            _, low_indices = flat.topk(k, largest=False)
            flat[low_indices] = 0

        seq_len = attn.shape[-1]
        fused = flat.view(seq_len, seq_len)

        # Add identity for residual, normalize
        identity = torch.eye(seq_len, dtype=fused.dtype)
        a = (fused + identity) / 2
        a = a / (a.sum(dim=-1, keepdim=True) + 1e-12)

        if result is None:
            result = a
        else:
            result = a @ result

    return result.numpy()

In [ ]:

# %% Strategy 3: Vanilla Rollout (baseline — kept for comparison)
def compute_vanilla_rollout(attention_maps_dict, layer_indices):
    """Original vanilla rollout — known to be diffuse. Kept for comparison."""
    rollout = None
    for layer_idx in layer_indices:
        attn = attention_maps_dict[layer_idx].squeeze(0)
        if attn.dtype == torch.bfloat16:
            attn = attn.to(torch.float32)
        attn = attn.mean(dim=0)
        attn = attn / (attn.sum(dim=-1, keepdim=True) + 1e-12)
        seq_len = attn.shape[0]
        identity = torch.eye(seq_len, dtype=attn.dtype)
        attn = 0.5 * attn + 0.5 * identity
        attn = attn / (attn.sum(dim=-1, keepdim=True) + 1e-12)
        if rollout is None:
            rollout = attn
        else:
            rollout = attn @ rollout
    return rollout.numpy()



In [ ]:

# %% Build spatial grid from any rollout/attention matrix
def build_grid_from_matrix(matrix, inputs, processor, target_word,
                            target_idx=None):
    """
    Maps a (seq_len, seq_len) rollout matrix to a spatial heatmap.
    Extracts the target token's row and assembles image-token columns
    into a spatial grid matching the original image tiling.
    """
    input_ids = inputs["input_ids"][0].tolist()

    if target_idx is None:
        tokens = processor.tokenizer.convert_ids_to_tokens(input_ids)
        target_idx = -1
        for i, t in enumerate(tokens):
            if target_word.lower() in t.lower():
                target_idx = i
                break
        if target_idx == -1:
            target_idx = len(input_ids) - 1

    # Find image token indices
    most_common_token = Counter(input_ids).most_common(1)[0][0]
    image_token_indices = [
        i for i, token in enumerate(input_ids) if token == most_common_token
    ]
    num_crops = inputs["pixel_values"].shape[1]
    tokens_per_crop = len(image_token_indices) // num_crops
    grid_size = int(np.sqrt(tokens_per_crop))

    num_local_crops = num_crops - 1
    crop_token_indices = [
        image_token_indices[c * tokens_per_crop : (c + 1) * tokens_per_crop]
        for c in range(num_local_crops)
    ]

    n_crop_cols = int(np.sqrt(num_local_crops))
    n_crop_rows = int(np.ceil(num_local_crops / n_crop_cols))
    full_grid_h = n_crop_rows * grid_size
    full_grid_w = n_crop_cols * grid_size

    word_attn = matrix[target_idx, :]

    full_grid = np.zeros((full_grid_h, full_grid_w), dtype=np.float32)
    for c, crop_indices in enumerate(crop_token_indices):
        crop_row = c // n_crop_cols
        crop_col = c % n_crop_cols
        crop_attn = word_attn[crop_indices]
        if isinstance(crop_attn, torch.Tensor):
            crop_attn = crop_attn.float().numpy()
        patch = crop_attn.reshape(grid_size, grid_size)
        full_grid[
            crop_row * grid_size : (crop_row + 1) * grid_size,
            crop_col * grid_size : (crop_col + 1) * grid_size,
        ] = patch

    return full_grid




In [ ]:
# %% Main extraction loop
print(f"\n{'='*60}")
print(f"Extracting attention for {len(frame_range)} frames...")
print(f"Strategies: Discard-Ratio, Gradient-Weighted, Vanilla")
print(f"{'='*60}\n")

# Accumulators per strategy
results = {
    "discard": {"grids": [], "means": []},
    "gradient": {"grids": [], "means": []},
    "vanilla": {"grids": [], "means": []},
}

reference_image = None
start_time = time.time()

for frame_num in tqdm(frame_range, desc="Processing frames"):
    frame_str = f"{frame_num:04d}"
    img_path = image_dir / f"frame{frame_str}.png"

    if not img_path.exists():
        print(f"  Skipping missing frame: {frame_str}")
        continue

    image = Image.open(img_path).convert("RGB")
    if reference_image is None:
        reference_image = image

    # === Strategy 1 & 3: Forward-only (discard + vanilla) ===
    attn_maps, inputs = get_attention_for_image(image, device)

    # Discard-ratio rollout
    discard_matrix = compute_discard_rollout(attn_maps, layer_indices,
                                              discard_ratio=0.9)
    discard_grid = build_grid_from_matrix(discard_matrix, inputs, processor,
                                           target_word)
    results["discard"]["grids"].append(discard_grid)
    results["discard"]["means"].append(float(discard_grid.mean()))

    # Vanilla rollout (for comparison)
    vanilla_matrix = compute_vanilla_rollout(attn_maps, layer_indices)
    vanilla_grid = build_grid_from_matrix(vanilla_matrix, inputs, processor,
                                           target_word)
    results["vanilla"]["grids"].append(vanilla_grid)
    results["vanilla"]["means"].append(float(vanilla_grid.mean()))

    del attn_maps

    # === Strategy 2: Gradient-weighted (needs backward pass) ===
    attn_maps_g, attn_grads, inputs_g, target_idx = \
        get_attention_with_gradients(image, target_word, device)

    grad_matrix = compute_grad_rollout(attn_maps_g, attn_grads, layer_indices,
                                        discard_ratio=0.9)
    grad_grid = build_grid_from_matrix(grad_matrix, inputs_g, processor,
                                        target_word, target_idx=target_idx)
    results["gradient"]["grids"].append(grad_grid)
    results["gradient"]["means"].append(float(grad_grid.mean()))

    del attn_maps_g, attn_grads, inputs, inputs_g
    torch.cuda.empty_cache()

elapsed = time.time() - start_time
n_frames = len(results["discard"]["grids"])
print(f"\nProcessed {n_frames} frames in {elapsed:.1f}s "
      f"({elapsed / n_frames:.2f}s/frame)")

# Stack
for key in results:
    results[key]["grids"] = np.stack(results[key]["grids"], axis=0)
    results[key]["mean_grid"] = results[key]["grids"].mean(axis=0)
    results[key]["std_grid"] = results[key]["grids"].std(axis=0)



In [ ]:

# ============================================================================
# VISUALIZATION
# ============================================================================

# %% Helper
def _normalise_for_display(grid):
    g = grid.copy()
    vmin, vmax = np.percentile(g, [2, 90])
    g = np.clip(g, vmin, vmax)
    g = (g - g.min()) / (g.max() - g.min() + 1e-8)
    return g


# %% Three-way comparison plot
def plot_three_way_comparison(results, pil_image, target_word, frame_idx=0,
                               save_path=None):
    """
    Side-by-side comparison of all three rollout strategies on a single frame.
    """
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    # Original
    axes[0].imshow(pil_image)
    axes[0].set_title("Original Image", fontsize=11, fontweight="bold")
    axes[0].axis("off")

    strategies = [
        ("vanilla", "Vanilla Rollout\n(diffuse baseline)", "Reds"),
        ("discard", "Discard-Ratio Rollout\n(top 10% kept)", "jet"),
        ("gradient", "Gradient-Weighted\nRollout", "jet"),
    ]

    for i, (key, title, cmap) in enumerate(strategies):
        grid = _normalise_for_display(results[key]["grids"][frame_idx])
        ax = axes[i + 1]
        ax.imshow(pil_image)
        ax.imshow(
            grid, cmap=cmap, alpha=0.6,
            extent=[0, pil_image.width, pil_image.height, 0],
        )
        ax.set_title(title, fontsize=10,
                      fontweight="bold" if key == "gradient" else "normal",
                      color="darkgreen" if key == "gradient" else "black")
        ax.axis("off")

    fig.suptitle(
        f"Rollout Strategy Comparison — '{target_word}' (frame {frame_idx})",
        fontsize=14, fontweight="bold", y=1.02,
    )
    plt.tight_layout()
    # if save_path:
    #     plt.savefig(save_path, dpi=150, bbox_inches="tight")
    #     print(f"Saved: {save_path}")
    plt.show()


plot_three_way_comparison(results, reference_image, target_word, frame_idx=0,
                           save_path="rollout_comparison_frame0.png")


In [ ]:
# %% Mean heatmaps — all three strategies
def plot_mean_heatmaps(results, pil_image, target_word, n_frames,
                        save_path=None):
    """Mean rollout heatmaps for each strategy, aggregated over all frames."""
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    axes[0].imshow(pil_image)
    axes[0].set_title("Original Image", fontsize=11, fontweight="bold")
    axes[0].axis("off")

    strategies = [
        ("vanilla", "Vanilla (mean)", "Reds"),
        ("discard", "Discard-Ratio (mean)", "jet"),
        ("gradient", "Gradient-Weighted (mean)", "jet"),
    ]

    for i, (key, title, cmap) in enumerate(strategies):
        grid = _normalise_for_display(results[key]["mean_grid"])
        ax = axes[i + 1]
        ax.imshow(pil_image)
        ax.imshow(
            grid, cmap=cmap, alpha=0.6,
            extent=[0, pil_image.width, pil_image.height, 0],
        )
        ax.set_title(title, fontsize=10)
        ax.axis("off")

    fig.suptitle(
        f"Mean Rollout Heatmaps — '{target_word}' (n={n_frames} frames)",
        fontsize=14, fontweight="bold", y=1.02,
    )
    plt.tight_layout()
    # if save_path:
    #     plt.savefig(save_path, dpi=150, bbox_inches="tight")
    #     print(f"Saved: {save_path}")
    plt.show()


plot_mean_heatmaps(results, reference_image, target_word, n_frames,
                    save_path="rollout_mean_comparison.png")


# %% Temporal comparison
def plot_temporal_comparison(results, target_word, save_path=None):
    """Temporal mean attention for all three strategies."""
    fig, ax = plt.subplots(figsize=(12, 5))

    colors = {"vanilla": "gray", "discard": "teal", "gradient": "crimson"}
    labels = {"vanilla": "Vanilla", "discard": "Discard-Ratio",
              "gradient": "Gradient-Weighted"}

    for key in ["vanilla", "discard", "gradient"]:
        vals = results[key]["means"]
        ax.plot(vals, color=colors[key], alpha=0.7, linewidth=1.2,
                label=labels[key])

        # Smoothed
        window = min(11, max(3, len(vals) // 10))
        if len(vals) >= window:
            smoothed = np.convolve(vals, np.ones(window) / window, mode="valid")
            offset = (window - 1) // 2
            ax.plot(
                range(offset, offset + len(smoothed)),
                smoothed, color=colors[key], linewidth=2.5, alpha=0.9,
                linestyle="--",
            )

    ax.set_xlabel("Frame", fontsize=12)
    ax.set_ylabel("Mean Rollout Attention", fontsize=12)
    ax.set_title(
        f"Temporal comparison — '{target_word}'",
        fontsize=14, fontweight="bold",
    )
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    # if save_path:
    #     plt.savefig(save_path, dpi=150, bbox_inches="tight")
    #     print(f"Saved: {save_path}")
    plt.show()


plot_temporal_comparison(results, target_word,
                          save_path="rollout_temporal_comparison.png")


# %% Quantitative metrics table
def compute_metrics(grid, label):
    flat = grid.flatten()
    flat_norm = flat / (flat.sum() + 1e-12)
    ent = scipy_entropy(flat_norm, base=2)
    peak = np.unravel_index(grid.argmax(), grid.shape)
    sorted_vals = np.sort(flat_norm)
    n = len(sorted_vals)
    idx = np.arange(1, n + 1)
    gini = np.sum((2 * idx - n - 1) * sorted_vals) / (n * np.sum(sorted_vals) + 1e-12)
    return {
        "Strategy": label,
        "Mean": f"{grid.mean():.6f}",
        "Max": f"{grid.max():.6f}",
        "Peak (r,c)": f"({peak[0]},{peak[1]})",
        "Entropy (bits)": f"{ent:.2f}",
        "Gini": f"{gini:.4f}",
    }


rows = [
    compute_metrics(results["vanilla"]["mean_grid"], "Vanilla"),
    compute_metrics(results["discard"]["mean_grid"], "Discard-Ratio (0.9)"),
    compute_metrics(results["gradient"]["mean_grid"], "Gradient-Weighted"),
]
metrics_df = pd.DataFrame(rows)
print(f"\n{'='*70}")
print(f"Rollout Metrics Comparison (n={n_frames} frames)")
print(f"{'='*70}")
print(metrics_df.to_string(index=False))
print()
# metrics_df.to_csv("rollout_metrics.csv", index=False)
print("Saved: rollout_metrics.csv")


# # %% Save results
# np.savez_compressed(
#     "rollout_grids.npz",
#     **{f"{k}_grids": v["grids"] for k, v in results.items()},
#     **{f"{k}_mean": v["mean_grid"] for k, v in results.items()},
#     target_word=target_word,
#     suite_name=suite_name,
#     n_frames=n_frames,
# )
# print("Saved: rollout_grids.npz")


# %% Summary
print(f"\n{'='*60}")
print("Attention Rollout analysis complete!")
print(f"  Frames processed : {n_frames}")
print(f"  Layers used      : {len(layer_indices)}")
print(f"  Target word      : '{target_word}'")
print(f"  Strategies       : Vanilla, Discard-Ratio, Gradient-Weighted")
print(f"{'='*60}")
print("\nOutputs:")
print("  - rollout_comparison_frame0.png  (single-frame 3-way comparison)")
print("  - rollout_mean_comparison.png    (mean heatmaps all strategies)")
print("  - rollout_temporal_comparison.png")
print("  - rollout_metrics.csv")
print("  - rollout_grids.npz")
